In [11]:
import numpy as np
import torch
from torch.utils.data import DataLoader
import os
import sys
import matplotlib.pyplot as plt

os.chdir("/home/patrick/ansermodelling")

from models.FFNN_network import FFNN
from data.anser_dataset import *
from models.train import *
from models.model_wrappers import NNSolver
from models.eval import report_error_stats

In [2]:
train_loader, test_loader = make_dataloaders("data/dataset.npz", normalise_data = True)

In [3]:
model = FFNN(input_dim=8, output_dim=6, hidden_dims=[256,256,256])
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = torch.nn.MSELoss()

In [4]:
history = train(model, train_loader, test_loader, optimizer, epochs=200, loss_fn = loss_fn, print_losses = True)

epoch 	 LR 	 Training Loss 	 Test Loss 	 e_p test 	 e_n test
1 	 1.00e-03 	 0.0629 	  0.0364 	 63.0240 	 22.1165
2 	 1.00e-03 	 0.0312 	  0.0257 	 53.9083 	 17.5850
3 	 1.00e-03 	 0.0245 	  0.0227 	 51.4638 	 16.1046
4 	 1.00e-03 	 0.0207 	  0.0202 	 56.6625 	 14.8100
5 	 1.00e-03 	 0.0183 	  0.0173 	 52.6558 	 13.6471
6 	 1.00e-03 	 0.0165 	  0.0167 	 48.8569 	 13.2031
7 	 1.00e-03 	 0.0154 	  0.0151 	 43.5345 	 12.2809
8 	 1.00e-03 	 0.0145 	  0.0154 	 47.1521 	 12.8726
9 	 1.00e-03 	 0.0136 	  0.0147 	 50.4705 	 12.3387
10 	 1.00e-03 	 0.0129 	  0.0125 	 42.6212 	 11.1411
11 	 1.00e-03 	 0.0125 	  0.0125 	 47.1149 	 11.1079
12 	 1.00e-03 	 0.0119 	  0.0121 	 44.9523 	 10.7432
13 	 1.00e-03 	 0.0115 	  0.0118 	 43.9539 	 10.7798
14 	 1.00e-03 	 0.0112 	  0.0112 	 41.9977 	 10.2593
15 	 1.00e-03 	 0.0108 	  0.0117 	 42.8460 	 10.6127
16 	 1.00e-03 	 0.0105 	  0.0110 	 41.7796 	 9.9439
17 	 1.00e-03 	 0.0102 	  0.0112 	 40.8785 	 10.2452
18 	 1.00e-03 	 0.0099 	  0.0115 	 42.4937 	 10.

In [5]:
torch.save(model.state_dict(), "models/checkpoints/nn_normal_noposeloss_noscheduler.pt")

In [9]:
test_set = np.load("data/test_set.npz")
measurements = test_set["xs"]
poses = test_set["ys"]

In [12]:
nn = NNSolver(model)

In [13]:
%%time
poses_pred_nn, success_nn = nn.solve(measurements)

CPU times: user 384 ms, sys: 25 ms, total: 409 ms
Wall time: 87 ms


In [15]:
print("With normal (noposeloss, no scheduler)")
report_error_stats(poses_pred_nn,poses,success_nn)

With normal (noposeloss, no scheduler)
Mean pos error: 139, mean angle error: 47.7
Median pos error: 138, Median angle error: 30.6
95% pos error : 240 95% angle error: 149
LM success rate: 1
Convergence rate: 0
Mean pos error of converged: nan, mean angle error of converged: nan 


/home/patrick/ansermodelling/models/eval.py:30: RuntimeWarning: Mean of empty slice
  print(f"Mean pos error of converged: {ex_conv.mean():.3g}, mean angle error of converged: {en_conv.mean():.3g} ")
/home/patrick/ansermodelling/.venv/lib/python3.14/site-packages/numpy/_core/_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
